# 01 — Exploratory data analysis

Exploration of the 250,000 Lichess blitz games (September 2026) used to train the model. Run `python -m src.download` and `python -m src.dataset` first. Figures used in the report are saved to `report/figures/eda_*.png`.

**Leakage note:** ratings and usernames are loaded here only to *describe* the data and the target. Feature code (`src/features.py`) never reads them.

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import load_config
from src.dataset import raw_path, load_table
from src.evaluate import style_axes, BLUE, ORANGE, INK, INK_2, SURFACE, BLUE_RAMP
cfg = load_config()
FIG = ROOT / 'report' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
def save(fig, name):
    fig.patch.set_facecolor(SURFACE); fig.tight_layout(); fig.savefig(FIG / f'{name}.png', dpi=150)
raw = pd.read_parquet(raw_path(cfg, cfg['data']['n_games']))
table = load_table(cfg)
for c in ['WhiteElo', 'BlackElo']: raw[c] = raw[c].astype(int)
print(f'{len(raw):,} games, {len(table):,} player rows, {table.shape[1]} columns')

250,000 games, 500,000 player rows, 62 columns


## 1. Filtering
How many streamed games each filter removed (logged by `src/download.py`).

In [2]:
counts = json.loads((ROOT / 'report' / f"filter_counts_{cfg['data']['month']}_n{cfg['data']['n_games']}.json").read_text())
steps = {k: v for k, v in counts.items() if k not in ('month', 'n_games')}
pd.Series(steps).to_frame('games').assign(share=lambda d: (d.games / counts['seen']).round(4))

,games,share
seen,562581,1.0000
not_blitz_event,302489,0.5377
kept,250000,0.4444
excluded_termination,709,0.0013
too_short,8686,0.0154
base_time_out_of_range,697,0.0012


## 2. Time span
The monthly file is chronological, so the first 250k blitz games cover only a few hours.

In [3]:
ts = pd.to_datetime(raw.UTCDate + ' ' + raw.UTCTime, format='%Y.%m.%d %H:%M:%S')
print('first game', ts.min(), '| last game', ts.max(), '| span', ts.max() - ts.min())
ts.dt.floor('h').value_counts().sort_index()

first game 2026-09-01 00:00:00 | last game 2026-09-01 07:15:11 | span 0 days 07:15:11


2026-09-01 00:00:00    33396
2026-09-01 01:00:00    31747
2026-09-01 02:00:00    30733
2026-09-01 03:00:00    29997
2026-09-01 04:00:00    32031
2026-09-01 05:00:00    37617
2026-09-01 06:00:00    42927
2026-09-01 07:00:00    11552
Name: count, dtype: int64

## 3. The target: rating distribution

In [4]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(table.rating, bins=np.arange(400, 3150, 50), color=BLUE, edgecolor=SURFACE, linewidth=0.5)
ax.axvline(table.rating.mean(), color=ORANGE, linewidth=2, label=f'mean {table.rating.mean():.0f}')
style_axes(ax, 'Rating distribution (one row per player per game)', 'Lichess blitz rating', 'Rows')
ax.legend(frameon=False, fontsize=8); save(fig, 'eda_rating_distribution')
table.rating.describe().round(0)

count    500000.0
mean       1622.0
std         360.0
min         400.0
25%        1374.0
50%        1640.0
75%        1878.0
max        3060.0
Name: rating, dtype: float64

## 4. Matchmaking: opponents have similar ratings
Lichess pairs players of similar strength, so the opponent's behaviour also carries information about a player's rating. This motivates including the opponent's features in each row.

In [5]:
diff = raw.WhiteElo - raw.BlackElo
print(f'corr(WhiteElo, BlackElo) = {raw.WhiteElo.corr(raw.BlackElo):.3f}')
print(f'median |difference| = {diff.abs().median():.0f}, 90th percentile = {diff.abs().quantile(0.9):.0f}')
fig, ax = plt.subplots(figsize=(5.5, 4.8))
hb = ax.hexbin(raw.WhiteElo, raw.BlackElo, gridsize=60, cmap=BLUE_RAMP, mincnt=3, bins='log', linewidths=0)
style_axes(ax, 'White vs. Black rating in the same game', 'White rating', 'Black rating')
fig.colorbar(hb, ax=ax).set_label('Games (log)', fontsize=8); save(fig, 'eda_opponent_ratings')

corr(WhiteElo, BlackElo) = 0.953
median |difference| = 30, 90th percentile = 117


## 5. Time control
Players choose their time control, and the choice is related to rating.

In [6]:
tc = table.groupby(table.base_time.astype(int).astype(str) + '+' + table.increment.astype(int).astype(str))
tc_stats = tc.rating.agg(['size', 'mean', 'std']).sort_values('size', ascending=False).head(8).round(0)
tc_stats

,size,mean,std
180+0,216228,1713.0,346.0
300+0,139268,1536.0,343.0
180+2,92388,1628.0,358.0
300+3,48360,1439.0,334.0
240+0,948,1632.0,375.0
180+1,822,1995.0,519.0
300+2,696,1647.0,528.0
180+3,386,1684.0,405.0


## 6. Game length, results and termination

In [7]:
games = table[table.color == 1]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(games.plies, bins=np.arange(20, 260, 5), color=BLUE, edgecolor=SURFACE, linewidth=0.5)
style_axes(ax, 'Game length', 'Half-moves (plies)', 'Games'); save(fig, 'eda_game_length')
print(games.plies.describe().round(0).to_dict())
print('White result:', games.own_result_score.value_counts(normalize=True).round(3).to_dict())
print('Termination:', games.termination.value_counts(normalize=True).round(3).to_dict())
print('Ended in checkmate:', round(games.ended_in_checkmate.mean(), 3))

{'count': 250000.0, 'mean': 70.0, 'std': 30.0, 'min': 20.0, '25%': 48.0, '50%': 65.0, '75%': 88.0, 'max': 452.0}
White result: {1.0: 0.495, 0.0: 0.464, 0.5: 0.041}
Termination: {'normal': 0.75, 'time forfeit': 0.249, 'insufficient material': 0.0}
Ended in checkmate: 0.244


## 7. Annotations available
All games have clock times (`%clk`); only a minority have engine evaluations (`%eval`), which is why engine features are a separate experiment (feature set B).

In [8]:
print('has %clk :', round(raw.pgn.str.contains('%clk', regex=False).mean(), 3))
print('has %eval:', round(raw.pgn.str.contains('%eval', regex=False).mean(), 3))

has %clk : 1.0


has %eval: 0.098


## 8. Which features relate to rating?
Pearson correlation with the target (linear relationships only; permutation importance in `src/evaluate.py` gives the model's view).

In [9]:
num = table.select_dtypes('number').drop(columns=['rating'])
corr = num.corrwith(table.rating).dropna().sort_values(key=abs, ascending=False)
top = corr.head(15)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top.index[::-1], top.values[::-1], color=[BLUE if v > 0 else ORANGE for v in top.values[::-1]], height=0.6)
ax.axvline(0, color=INK_2, linewidth=0.8)
style_axes(ax, 'Strongest linear correlations with rating', 'Pearson r (blue positive, orange negative)', '')
ax.tick_params(axis='y', labelsize=8); save(fig, 'eda_correlations')
top.round(3)

opp_time_first10              -0.262
own_time_first10              -0.246
base_time                     -0.238
own_avg_time_per_move         -0.210
own_distinct_pieces_opening    0.207
opp_avg_time_per_move         -0.205
opp_distinct_pieces_opening    0.205
own_n_moves                    0.161
plies                          0.160
opp_n_moves                    0.160
own_castled                    0.145
opp_capture_rate              -0.143
opp_castled                    0.137
increment                     -0.127
own_capture_rate              -0.117
dtype: float64

In [10]:
fig, ax = plt.subplots(figsize=(7, 3.8))
bins = pd.cut(table.rating, [0, 1000, 1250, 1500, 1750, 2000, 2250, 4000])
m = table.groupby(bins, observed=True).own_time_first10.median()
ax.bar([str(b) for b in m.index], m.values, color=BLUE, width=0.6)
style_axes(ax, 'Stronger players play the opening faster', 'Rating bucket', 'Median seconds on first 10 moves')
ax.tick_params(axis='x', labelsize=7); save(fig, 'eda_opening_time')
m.round(1)

rating
(0, 1000]       42.0
(1000, 1250]    32.0
(1250, 1500]    26.0
(1500, 1750]    21.0
(1750, 2000]    18.0
(2000, 2250]    17.0
(2250, 4000]    17.0
Name: own_time_first10, dtype: float64

## 9. Players
The same user can play many games. Usernames are used **only** to measure this and for the grouped-by-player evaluation, never as a feature.

In [11]:
per_player = table.player.value_counts()
print(f'{per_player.size:,} unique players; games per player: median {per_player.median():.0f}, '
      f'mean {per_player.mean():.1f}, max {per_player.max()}')
train_players = set(table.loc[table.split == 'train', 'player'])
test = table[table.split == 'test']
print(f'test rows whose player also appears in training: {test.player.isin(train_players).mean():.1%}')

120,248 unique players; games per player: median 2, mean 4.2, max 99


test rows whose player also appears in training: 89.8%


## 10. Missing values
Material at moves 30/40 is missing when the game ended earlier; castling move is missing when a player never castled. Trees handle NaN natively; Ridge gets median imputation plus missing-indicator columns.

In [12]:
table.isna().mean().loc[lambda s: s > 0].sort_values(ascending=False).round(3)

own_material_m40    0.670
opp_material_m40    0.670
opp_material_m30    0.414
own_material_m30    0.414
opp_castle_move     0.182
own_castle_move     0.182
own_material_m20    0.144
opp_material_m20    0.144
dtype: float64